In [ ]:
# Ubiquitin family: mean GT error vs sequence identity and AF2 TM-score (Fig. S19), and shifts relative to WT (Fig. S20).
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from tmtools import tm_align

def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base, "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2, "legend.fontsize": base - 2, "svg.fonttype": "none"})

set_style(base=16)

FIG5 = Path("../../../data/analysis/Fig5")
PRED_PATH = FIG5 / "predictions_N_ensemble_10seed_sorted_centered_err.json"
GT_PATH = Path("../../../data/rBMRB/cluster_30p/PANAV/test.json")
AF2_DIR = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
FIG_DIR = Path("Figs")
FIG_DIR.mkdir(exist_ok=True)

REF_PID = "30693_1"    # plain WT ubiquitin, 76 aa
CORE_LEN = 76          # last 76 residues of every entry line up 1:1 with WT (only N-terminal tags differ)
UBIQUITIN_PIDS = [
    "11505_1", "11547_1", "15410_1", "16228_1", "17439_1", "17769_1", "18611_1", "18831_1",
    "19670_1", "25972_1", "27512_1", "30693_1", "34351_1", "4493_1", "4769_1", "51725_1",
    "52529_1", "5387_1", "6466_1",
]

plt.rcParams.update({"font.family": "Arial", "font.size": 16, "axes.titlesize": 16,
                     "axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14,
                     "legend.fontsize": 14, "svg.fonttype": "none"})
BLUE, RED, GREY, PURPLE = "#61AFEF", "#E06C75", "#999999", "#C678DD"


def fixed_axes(ax_w=4.6, ax_h=4.2, canvas=9.0):
    """Axes box exactly ax_w x ax_h inches; savefig(bbox_inches='tight') crops the canvas."""
    fig = plt.figure(figsize=(canvas, canvas))
    ax = fig.add_axes([1.5 / canvas, 1.5 / canvas, ax_w / canvas, ax_h / canvas])
    return fig, ax


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved", name)


# ---------------- data ----------------
pred_json = json.load(open(PRED_PATH))
gt_json = json.load(open(GT_PATH))
inv = {v: k for k, v in gt_json["entry_ID"].items()}


def valid_arrays(pid):
    """pred |error|, GT |error| (oof_resid) with mask & oof_mask & finite -> NaN elsewhere."""
    p = np.array(pred_json["pred"][pid], dtype=np.float64)
    g = np.array(pred_json["oof_resid"][pid], dtype=np.float64)
    m = np.array(pred_json["mask"][pid], dtype=bool) & np.array(pred_json["oof_mask"][pid], dtype=bool)
    n = min(len(p), len(g), len(m))
    p, g, m = p[:n], g[:n], m[:n] & np.isfinite(p[:n]) & np.isfinite(g[:n])
    return np.where(m, p, np.nan), np.where(m, g, np.nan)


# outlier status: same IQR rule on the cross-protein (mean pred, mean GT) fit as the main analysis
all_pids = [pid for pid in pred_json["pred"] if np.isfinite(valid_arrays(pid)[1]).any()]
mp = np.array([np.nanmean(valid_arrays(pid)[0]) for pid in all_pids])
mg = np.array([np.nanmean(valid_arrays(pid)[1]) for pid in all_pids])
b1, b0 = np.polyfit(mp, mg, 1)
resid = mg - (b1 * mp + b0)
q1, q3 = np.percentile(resid, [25, 75])
lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
UNDER = {pid for pid, r in zip(all_pids, resid) if r > hi}
OVER = {pid for pid, r in zip(all_pids, resid) if r < lo}


def status_color(pid):
    return RED if pid in UNDER else PURPLE if pid in OVER else GREY


def gt_shift_core(pid):
    """GT 15N shifts of the 76-aa core (NaN where unassigned)."""
    s = np.array([np.nan if v is None else v for v in gt_json["N"][inv[pid]]], dtype=np.float64)
    return s[-CORE_LEN:]


# ======== Fig. S19: GT error vs seq identity (left) and AF2 TM-score (right) ========
_aligner = PairwiseAligner()
_aligner.mode = "global"
_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
_aligner.open_gap_score, _aligner.extend_gap_score = -10, -0.5


def seq_identity(seq, ref):
    aln = _aligner.align(seq, ref)[0]
    n_ident = sum(seq[i] == ref[j] for (s1, e1), (s2, e2) in zip(*aln.aligned)
                  for i, j in zip(range(s1, e1), range(s2, e2)))
    return n_ident / len(ref)


def af2_ca(pid):
    chain = next(PDBParser(QUIET=True).get_structure(pid, AF2_DIR / f"{pid}.pdb")[0].get_chains())
    res = [r for r in chain if r.id[0] == " " and "CA" in r]
    return (np.array([r["CA"].coord for r in res], dtype=np.float64),
            "".join(index_to_one(three_to_index(r.get_resname())) for r in res))


ref_seq = "".join(gt_json["sequence"][inv[REF_PID]])
ref_xyz, ref_ca_seq = af2_ca(REF_PID)
rows = []
for pid in UBIQUITIN_PIDS:
    xyz, ca_seq = af2_ca(pid)
    rows.append((pid, np.nanmean(valid_arrays(pid)[1]),
                 seq_identity("".join(gt_json["sequence"][inv[pid]]), ref_seq),
                 tm_align(xyz, ref_xyz, ca_seq, ref_ca_seq).tm_norm_chain2))
pids, gerr, ident, tms = map(np.array, zip(*rows))
colors = [status_color(p) for p in pids]
for r in sorted(rows, key=lambda r: r[1]):
    print(f"{r[0]:8s}  GT err={r[1]:.2f}  identity={r[2]:.3f}  TM(AF2 vs WT AF2)={r[3]:.3f}")

In [ ]:
# ======== Fig. S20, WT-referenced view: does the prediction follow the GT change? ========
# For every ubiquitin entry and core position: dGT = GT - GT(WT), dPred = pred - pred(WT), with pred the
# 10-seed SLAE-CS ensemble mean. Errors the model already makes on WT cancel out of both axes, so only the
# entry-vs-WT difference is left. Slope 1 (y = x) = the model reproduces the GT difference; slope 0 (y = 0)
# = it predicts the WT shift regardless, i.e. the input structure does not carry the difference.
from scipy.stats import linregress, pearsonr
set_style(base=16)
ENS_PATH = Path("../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json")
ALIAS_PPM = 15.0      # |dGT| above this = spectral-aliasing deposition artifact (e.g. 27512 T9 / S20), dropped
ens_json = json.load(open(ENS_PATH))


def pred_core(pid):
    n = len(gt_json["sequence"][inv[pid]])
    mean = np.array([np.nan if v is None else v for v in ens_json["mean"][pid][:n]], dtype=np.float64)
    mean[~np.array(ens_json["mask"][pid][:n], dtype=bool)] = np.nan
    return mean[-CORE_LEN:]


gt_wt, pred_wt = gt_shift_core(REF_PID), pred_core(REF_PID)
dG, dP, cc = [], [], []
for pid in UBIQUITIN_PIDS:
    if pid == REF_PID:
        continue
    d_g, d_p = gt_shift_core(pid) - gt_wt, pred_core(pid) - pred_wt
    ok = np.isfinite(d_g) & np.isfinite(d_p) & (np.abs(d_g) < ALIAS_PPM)
    dG.append(d_g[ok]); dP.append(d_p[ok]); cc += [status_color(pid)] * int(ok.sum())
dG, dP, cc = np.concatenate(dG), np.concatenate(dP), np.array(cc)
is_normal = cc == GREY                                   # fit on non-outlier entries only
fit = linregress(dG[is_normal], dP[is_normal])
print(f"{len(dG)} residues ({int(is_normal.sum())} non-outlier) | non-outlier slope dPred~dGT = "
      f"{fit.slope:.2f} (r = {fit.rvalue:.2f})")
print(f"{len(dG)} residues | slope dPred~dGT = {fit.slope:.2f} (r = {fit.rvalue:.2f}); "
      f"residual error change = -(1 - slope) x dGT")

fig, ax = fixed_axes()
lim = np.nanmax(np.abs(np.concatenate([dG, dP]))) * 1.05
ax.plot([-lim, lim], [-lim, lim], "k--", lw=1, zorder=1, label="y = x (prediction follows GT)")
ax.axhline(0, color="0.5", lw=1, ls=":", zorder=1, label="y = 0 (prediction stays at WT)")
for c, lab in [(GREY, "non-outlier"), (RED, "under-predicted")]:
    m = cc == c
    if m.any():
        ax.scatter(dG[m], dP[m], s=30 if c != GREY else 18, alpha=.85 if c != GREY else .45,
                   color=c, edgecolor="none", zorder=3 if c != GREY else 2, label=lab)
xs = np.array([-lim, lim])
r_normal = pearsonr(dG[is_normal], dP[is_normal])[0]
ax.plot(xs, fit.intercept + fit.slope * xs, color=BLUE, lw=2, zorder=4,
        label=f"fit (non-outlier), r = {r_normal:.2f}")
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("Others GT - WT GT $^{15}$N shift (ppm)")
ax.set_ylabel("Others prediction - WT prediction $^{15}$N shift (ppm)")
ax.set_title(f"{len(UBIQUITIN_PIDS) - 1} ubiquitin entries vs WT {REF_PID.split('_')[0]}, {len(dG)} residues")
ax.legend(frameon=False, loc="upper left", fontsize=13)
save(fig, "ubiquitin_dpred_vs_dgt")
plt.show()


In [ ]:
# side-by-side panels sharing y (mean GT error): left = sequence identity, right = AF2 TM-score
set_style(base=17)
canvas = 9.0
AX_H, W_L, W_R, GAP = 4.2, 2.8, 1.6, 0.3
fig = plt.figure(figsize=(canvas, canvas))
ax_l = fig.add_axes([1.5 / canvas, 1.5 / canvas, W_L / canvas, AX_H / canvas])
ax_r = fig.add_axes([(1.5 + W_L + GAP) / canvas, 1.5 / canvas, W_R / canvas, AX_H / canvas],
                    sharey=ax_l)

# left: sequence identity, with WT and outliers labeled
ax_l.scatter(ident * 100, gerr, s=75, marker="D", c=colors, edgecolor="k", lw=.8, zorder=3)
is_wt = pids == REF_PID   # WT on top with a thick blue edge so it stands out in the 100% stack
ax_l.scatter(ident[is_wt] * 100, gerr[is_wt], s=75, marker="D", c=GREY, edgecolor=BLUE, lw=2.2, zorder=4)
ax_l.set_xlim(min(ident) * 100 - 2, 101)
ax_l.xaxis.set_major_locator(plt.MultipleLocator(2))   # integer ticks only
ax_l.set_xlabel("sequence identity (%)")
ax_l.set_ylabel("mean ground-truth |error| per protein (ppm)")
ax_l.set_ylim(0.75, gerr.max() + 0.2)
# hand-placed label offsets (points); WT and outliers only
LABEL_OFFSETS = {
    REF_PID:   (-40, 30),
    "18611_1": (-14, 0),
    "11547_1": (-14, 0),
    "27512_1": (-14, -10),
    # "17439_1": (-14, -24),
}
for pid, x, y in zip(pids, ident * 100, gerr):
    if pid not in LABEL_OFFSETS:
        continue
    dx, dy = LABEL_OFFSETS[pid]
    txt = f"WT {pid.split('_')[0]}" if pid == REF_PID else pid.split("_")[0]
    ax_l.annotate(txt, (x, y), xytext=(dx, dy), textcoords="offset points",
                  ha="right" if dx < 0 else "left", va="center", fontsize=14,
                  arrowprops=dict(arrowstyle="-", color="grey", lw=.6, shrinkB=5))

# right: AF2 TM-score
ax_r.axvspan(tms.min(), 1.0, color=BLUE, alpha=.15, lw=0, zorder=1)
ax_r.scatter(tms, gerr, s=60, marker="o", c=colors, edgecolor="k", lw=.8, zorder=3)
ax_r.scatter(tms[is_wt], gerr[is_wt], s=60, marker="o", c=GREY, edgecolor=BLUE, lw=2.2, zorder=4)
ax_r.set_xlim(0.9, 1.01)
ax_r.set_xticks([0.9, 0.95, 1.0])
ax_r.set_xlabel("AF2 TM-score")
ax_r.text(0.05, 0.98, f"all TM \u2265 {tms.min():.3f}", transform=ax_r.transAxes,
          ha="left", va="top", fontsize=15)
ax_r.tick_params(labelleft=False)

handles = [Line2D([], [], ls="none", marker="s", ms=8, mfc=c, mec="none", label=l)
           for c, l in [(GREY, "non-outlier"), (RED, "under-predicted")]]
fig.legend(handles=handles, frameon=False, loc="lower center",
           bbox_to_anchor=((1.5 + (W_L + GAP + W_R) / 2) / canvas, (1.5 + AX_H) / canvas),
           borderaxespad=0.1, ncol=2, columnspacing=0.8, handletextpad=0.3, handlelength=1.0)
fig.text((1.5 + (W_L + GAP + W_R) / 2) / canvas, (1.5 + AX_H + 0.38) / canvas,
         f"Identity and TM-score both vs WT {REF_PID.split('_')[0]}", ha="center", va="bottom", fontsize=16)
save(fig, "ubiquitin_err_vs_seqid_tmscore_sharey")
plt.show()
